In [1]:
# Cell 1 — Imports, paths, and reconstruction settings
#
# Purpose:
# Reconstruct the P28 cell using the original Step07 procedure.
# Save the numerical results for separate evaluation and plotting.

from pathlib import Path
from time import perf_counter
import json
import sys

import numpy as np
import pandas as pd

from scipy import ndimage as ndi
from skimage.graph import MCP_Geometric


# --------------------------------------------------
# 1. Project paths
# --------------------------------------------------

def find_project_root(start=None):
    start_path = Path.cwd() if start is None else Path(start)
    start_path = start_path.resolve()

    for directory in (start_path, *start_path.parents):
        if (
            (directory / "README.md").is_file()
            and (directory / "notebooks").is_dir()
            and (directory / "data").is_dir()
        ):
            return directory

    raise FileNotFoundError(
        "Run this notebook from within the repository containing "
        "README.md, notebooks/, and data/."
    )


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT


def project_relative(path):
    return Path(path).relative_to(PROJECT_ROOT).as_posix()

SAMPLE_NAME = "p28_330_HET_009"

DATA_DIR = DATA_ROOT / "data" / SAMPLE_NAME

INPUT_DIR = (
    DATA_ROOT
    / "results"
    / SAMPLE_NAME
    / "05_soma_target"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / SAMPLE_NAME
    / "07_reconstruction"
)


# --------------------------------------------------
# 2. Active-domain and transition-weight settings
# --------------------------------------------------

# Retain the threshold used in the original Step07 notebook.
ACTIVE_THRESHOLD = 0.12

SIGNAL_POWER = 5.0
NUMERICAL_EPSILON = 1e-12

TARGET_DILATION_ITERATIONS = 2
CONNECTIVITY_3D = np.ones((3, 3, 3), dtype=bool)


# --------------------------------------------------
# 3. Hitting-probability solver settings
# --------------------------------------------------

SOLVER_MODE = "hook_friendly"
OUTWARD_TOLERANCE_XY = 1.0

MAXIMUM_ITERATIONS = 40_000
CONVERGENCE_TOLERANCE = 1e-5
CONVERGENCE_CHECK_INTERVAL = 10
PROGRESS_INTERVAL = 100


# --------------------------------------------------
# 4. Path-length correction and mask filtering
# --------------------------------------------------

MINIMUM_PATH_LENGTH = 1.0

SCORE_THRESHOLD = 0.50
INTENSITY_THRESHOLD = 0.12

RAW_MIP_QUANTILES = (0.01, 0.998)

MIN_COMPONENT_VOXELS = 10


# --------------------------------------------------
# 5. Check the Step05 input files
# --------------------------------------------------

INPUT_FILES = {
    "metadata": INPUT_DIR / "roi_metadata.csv",
    "signal": INPUT_DIR / "mg_probability_roi_yxz_float32.bin",
    "target": INPUT_DIR / "soma_target_roi_yxz_uint8.bin",
    "raw_mip": INPUT_DIR / "raw_mip_roi_yx_float32.bin",
}

missing_files = [
    project_relative(path)
    for path in INPUT_FILES.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing input files:\n" + "\n".join(missing_files)
    )


# --------------------------------------------------
# 6. Import the existing solver
# --------------------------------------------------

solver_file = PROJECT_ROOT / "microglia_hitting" / "solver.py"

if not solver_file.is_file():
    raise FileNotFoundError(f"Solver not found: {project_relative(solver_file)}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from microglia_hitting import solve_hitting_probability

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample: {SAMPLE_NAME}")
print(f"Active threshold: {ACTIVE_THRESHOLD}")
print(f"Solver mode: {SOLVER_MODE}")
print(f"Score threshold: {SCORE_THRESHOLD}")
print(f"Raw-MIP intensity threshold: {INTENSITY_THRESHOLD}")
print(f"Minimum component size: {MIN_COMPONENT_VOXELS} voxels")
print(f"Output directory: {project_relative(OUTPUT_DIR)}")

Sample: p28_330_HET_009
Active threshold: 0.12
Solver mode: hook_friendly
Score threshold: 0.5
Raw-MIP intensity threshold: 0.12
Minimum component size: 10 voxels
Output directory: results/p28_330_HET_009/07_reconstruction


In [2]:
# Cell 2 — Load the Step05 inputs and verify their dimensions

# --------------------------------------------------
# 1. Read ROI metadata
# --------------------------------------------------

metadata_table = pd.read_csv(INPUT_FILES["metadata"])

if len(metadata_table) != 1:
    raise ValueError("Expected one row of ROI metadata.")

metadata = metadata_table.iloc[0]

shape_yxz = (
    int(metadata["dim_y"]),
    int(metadata["dim_x"]),
    int(metadata["dim_z"]),
)

shape_zyx = (
    shape_yxz[2],
    shape_yxz[0],
    shape_yxz[1],
)


# --------------------------------------------------
# 2. Read the exported arrays in R storage order
# --------------------------------------------------

signal_yxz = np.fromfile(
    INPUT_FILES["signal"],
    dtype="<f4",
).reshape(
    shape_yxz,
    order="F",
)

target_yxz = np.fromfile(
    INPUT_FILES["target"],
    dtype=np.uint8,
).reshape(
    shape_yxz,
    order="F",
).astype(bool)

raw_mip_yx = np.fromfile(
    INPUT_FILES["raw_mip"],
    dtype="<f4",
).reshape(
    shape_yxz[:2],
    order="F",
)


# --------------------------------------------------
# 3. Convert the 3D arrays from Y-X-Z to Z-Y-X
# --------------------------------------------------

signal_zyx = np.moveaxis(signal_yxz, 2, 0)
target_zyx = np.moveaxis(target_yxz, 2, 0)


# --------------------------------------------------
# 4. Read the soma center in local ROI coordinates
# --------------------------------------------------

soma_center_x_local = float(metadata["soma_center_x_local"])
soma_center_y_local = float(metadata["soma_center_y_local"])


# --------------------------------------------------
# 5. Check dimensions and input values
# --------------------------------------------------

assert signal_zyx.shape == shape_zyx
assert target_zyx.shape == shape_zyx
assert raw_mip_yx.shape == shape_yxz[:2]

assert np.isfinite(signal_zyx).all(), "Non-finite input signal."
assert np.isfinite(raw_mip_yx).all(), "Non-finite raw-MIP intensities."
assert target_zyx.any(), "The soma target is empty."

print(f"Original shape (Y, X, Z): {shape_yxz}")
print(f"Working shape (Z, Y, X):  {shape_zyx}")
print(f"Raw-MIP shape (Y, X):     {raw_mip_yx.shape}")

print(
    "Soma center (x, y): "
    f"({soma_center_x_local:.3f}, {soma_center_y_local:.3f})"
)

print(f"Soma-target voxels: {int(target_zyx.sum()):,}")

print(
    "Signal range: "
    f"[{signal_zyx.min():.6f}, {signal_zyx.max():.6f}]"
)

print(
    "Raw-MIP intensity range: "
    f"[{raw_mip_yx.min():.6f}, {raw_mip_yx.max():.6f}]"
)

Original shape (Y, X, Z): (241, 241, 161)
Working shape (Z, Y, X):  (161, 241, 241)
Raw-MIP shape (Y, X):     (241, 241)
Soma center (x, y): (120.754, 120.434)
Soma-target voxels: 2,829
Signal range: [0.000000, 1.000000]
Raw-MIP intensity range: [0.007279, 0.062486]


In [3]:
# Cell 3 — Construct the soma-connected active domain

# --------------------------------------------------
# 1. Threshold the preprocessed signal
# --------------------------------------------------

thresholded_zyx = signal_zyx >= ACTIVE_THRESHOLD


# --------------------------------------------------
# 2. Label the 26-connected components
# --------------------------------------------------

component_labels_zyx, component_count = ndi.label(
    thresholded_zyx,
    structure=CONNECTIVITY_3D,
)


# --------------------------------------------------
# 3. Expand the target for the contact test
# --------------------------------------------------

expanded_target_zyx = ndi.binary_dilation(
    target_zyx,
    structure=CONNECTIVITY_3D,
    iterations=TARGET_DILATION_ITERATIONS,
)


# --------------------------------------------------
# 4. Identify all components touching the soma neighborhood
# --------------------------------------------------

soma_component_ids = np.unique(
    component_labels_zyx[
        expanded_target_zyx & thresholded_zyx
    ]
)

soma_component_ids = soma_component_ids[
    soma_component_ids > 0
]

if soma_component_ids.size == 0:
    raise RuntimeError(
        "No thresholded component touches the expanded soma target. "
        "Check the active threshold and target coordinates."
    )


# --------------------------------------------------
# 5. Retain all touching components and the original target
# --------------------------------------------------

active_zyx = np.isin(
    component_labels_zyx,
    soma_component_ids,
)

# The expanded target is used only for identifying contact.
# Only the original soma target is explicitly added.
active_zyx |= target_zyx

assert active_zyx.shape == signal_zyx.shape
assert np.all(active_zyx[target_zyx])


# --------------------------------------------------
# 6. Record the active-domain summary
# --------------------------------------------------

active_domain_summary = {
    "active_threshold": float(ACTIVE_THRESHOLD),
    "target_dilation_iterations": int(TARGET_DILATION_ITERATIONS),
    "component_policy": "all_touching_components",
    "thresholded_components": int(component_count),
    "retained_components": int(soma_component_ids.size),
    "thresholded_voxels": int(thresholded_zyx.sum()),
    "active_voxels": int(active_zyx.sum()),
    "target_voxels": int(target_zyx.sum()),
    "unknown_voxels": int((active_zyx & ~target_zyx).sum()),
}

print(f"Active threshold: {ACTIVE_THRESHOLD}")
print(f"Thresholded components: {component_count:,}")
print(f"Retained components: {soma_component_ids.size:,}")
print(f"Thresholded voxels: {thresholded_zyx.sum():,}")
print(f"Active-domain voxels: {active_zyx.sum():,}")
print(f"Soma-target voxels: {target_zyx.sum():,}")

Active threshold: 0.12
Thresholded components: 4,436
Retained components: 1
Thresholded voxels: 1,612,883
Active-domain voxels: 1,606,031
Soma-target voxels: 2,829


In [4]:
# Cell 4 — Compute and save the hitting-probability map

import re
from contextlib import redirect_stdout
from tqdm.auto import tqdm


# --------------------------------------------------
# 1. Connect the solver's progress messages to tqdm
# --------------------------------------------------

class SolverProgress:
    def __init__(self, bar):
        self.bar = bar
        self.buffer = ""

    def write(self, text):
        self.buffer += text

        while "\n" in self.buffer:
            line, self.buffer = self.buffer.split("\n", 1)

            match = re.search(
                r"iteration=([\d,]+); max_change=([\d.eE+-]+)",
                line,
            )

            if match:
                iteration = int(match.group(1).replace(",", ""))

                self.bar.update(iteration - self.bar.n)

                self.bar.set_postfix(
                    max_change=match.group(2),
                    refresh=False,
                )

        return len(text)

    def flush(self):
        pass


# --------------------------------------------------
# 2. Prepare transition weights
# --------------------------------------------------

transition_weights_zyx = (
    np.clip(signal_zyx, 0.0, 1.0) ** SIGNAL_POWER
    + NUMERICAL_EPSILON
)


# --------------------------------------------------
# 3. Call the shared solver
# --------------------------------------------------

with tqdm(
    total=MAXIMUM_ITERATIONS,
    desc="Solving P_hit",
    unit="iter",
) as progress:

    with redirect_stdout(SolverProgress(progress)):
        P_hit_zyx, solver_info = solve_hitting_probability(
            weights=transition_weights_zyx,
            active=active_zyx,
            target=target_zyx,
            soma_x=soma_center_x_local,
            soma_y=soma_center_y_local,
            mode=SOLVER_MODE,
            outward_tolerance_xy=OUTWARD_TOLERANCE_XY,
            maximum_iterations=MAXIMUM_ITERATIONS,
            convergence_tolerance=CONVERGENCE_TOLERANCE,
            convergence_check_interval=CONVERGENCE_CHECK_INTERVAL,
            progress_interval=PROGRESS_INTERVAL,
            verbose=True,
        )

    progress.update(solver_info["iterations"] - progress.n)

    progress.set_description(
        "Converged"
        if solver_info["converged"]
        else "Iteration limit reached"
    )


# --------------------------------------------------
# 4. Save the solver record and check convergence
# --------------------------------------------------

(OUTPUT_DIR / "solver_info.json").write_text(
    json.dumps(solver_info, indent=2),
    encoding="utf-8",
)

if not solver_info["converged"]:
    raise RuntimeError(
        "The hitting-probability solver did not converge. "
        "Review solver_info.json before continuing."
    )

assert P_hit_zyx.shape == signal_zyx.shape
assert np.isfinite(P_hit_zyx).all()
assert np.allclose(P_hit_zyx[target_zyx], 1.0)
assert np.all(P_hit_zyx[~active_zyx] == 0.0)


# --------------------------------------------------
# 5. Save the converged probability map
# --------------------------------------------------

np.save(
    OUTPUT_DIR / "P_hit_zyx.npy",
    P_hit_zyx,
)

print(f"Iterations: {solver_info['iterations']:,}")
print(f"Final maximum change: {solver_info['final_maximum_change']:.3e}")
print(f"Solver runtime: {solver_info['runtime_minutes']:.2f} minutes")
print(f"Saved: {project_relative(OUTPUT_DIR / 'P_hit_zyx.npy')}")

Solving P_hit:   0%|          | 0/40000 [00:00<?, ?iter/s]

Iterations: 18,020
Final maximum change: 9.987e-06
Solver runtime: 13.45 minutes
Saved: results/p28_330_HET_009/07_reconstruction/P_hit_zyx.npy


In [5]:
# Cell 5 — Compute and save the 3D geodesic path lengths

if not solver_info["converged"]:
    raise RuntimeError(
        "The hitting map has not converged. "
        "Review the solver result before continuing."
    )

start_time = perf_counter()


# --------------------------------------------------
# 1. Define traversal costs
# --------------------------------------------------

# Unit cost inside the active domain; no traversal outside it.
geodesic_cost_zyx = np.where(
    active_zyx,
    1.0,
    np.inf,
).astype(np.float32)

# Use every soma-target voxel as a starting point.
soma_start_coordinates = [
    tuple(coordinate)
    for coordinate in np.argwhere(target_zyx)
]


# --------------------------------------------------
# 2. Compute geometric shortest-path lengths
# --------------------------------------------------

# Use all 26 neighbors and geometric step lengths in voxel units.
L_raw_zyx, _ = MCP_Geometric(
    geodesic_cost_zyx,
    fully_connected=True,
).find_costs(
    starts=soma_start_coordinates,
)


# --------------------------------------------------
# 3. Apply the minimum effective path length
# --------------------------------------------------

reachable_zyx = active_zyx & np.isfinite(L_raw_zyx)

L_zyx = np.full(
    shape_zyx,
    np.inf,
    dtype=np.float32,
)

L_zyx[reachable_zyx] = np.maximum(
    L_raw_zyx[reachable_zyx],
    MINIMUM_PATH_LENGTH,
).astype(np.float32)

L_zyx[target_zyx] = MINIMUM_PATH_LENGTH

assert L_zyx.shape == signal_zyx.shape
assert np.all(L_zyx[reachable_zyx] >= MINIMUM_PATH_LENGTH)
assert np.allclose(L_zyx[target_zyx], MINIMUM_PATH_LENGTH)


# --------------------------------------------------
# 4. Record and save the result
# --------------------------------------------------

path_length_info = {
    "array_order": "Z,Y,X",
    "neighbor_connectivity": 26,
    "distance_units": "voxels",
    "minimum_effective_length": float(MINIMUM_PATH_LENGTH),
    "soma_start_voxels": len(soma_start_coordinates),
    "reachable_active_voxels": int(reachable_zyx.sum()),
    "unreachable_active_voxels": int(
        active_zyx.sum() - reachable_zyx.sum()
    ),
    "maximum_effective_length": float(L_zyx[reachable_zyx].max()),
    "runtime_minutes": (perf_counter() - start_time) / 60.0,
}

np.save(
    OUTPUT_DIR / "L_zyx.npy",
    L_zyx,
)

(OUTPUT_DIR / "path_length_info.json").write_text(
    json.dumps(path_length_info, indent=2),
    encoding="utf-8",
)

print(f"Soma starting voxels: {len(soma_start_coordinates):,}")
print(f"Reachable active voxels: {reachable_zyx.sum():,}")
print(
    "Unreachable active voxels: "
    f"{path_length_info['unreachable_active_voxels']:,}"
)
print(
    "Maximum effective path length: "
    f"{path_length_info['maximum_effective_length']:.3f} voxels"
)
print(f"Saved: {project_relative(OUTPUT_DIR / 'L_zyx.npy')}")

Soma starting voxels: 2,829
Reachable active voxels: 1,606,031
Unreachable active voxels: 0
Maximum effective path length: 178.217 voxels
Saved: results/p28_330_HET_009/07_reconstruction/L_zyx.npy


In [6]:
# Cell 6 — Compute and save the path-length-adjusted score

# --------------------------------------------------
# 1. Identify voxels with a valid probability and path length
# --------------------------------------------------

valid_score_voxels = (
    active_zyx
    & np.isfinite(L_zyx)
    & (P_hit_zyx > 0)
)


# --------------------------------------------------
# 2. Calculate the score using float64 intermediate values
# --------------------------------------------------

sqrtL_score_zyx = np.zeros(
    shape_zyx,
    dtype=np.float32,
)

sqrtL_score_zyx[valid_score_voxels] = np.exp(
    np.log(
        P_hit_zyx[valid_score_voxels].astype(np.float64)
    )
    / np.sqrt(
        L_zyx[valid_score_voxels].astype(np.float64)
    )
).astype(np.float32)

# Retain the maximum score at the soma target.
sqrtL_score_zyx[target_zyx] = 1.0


# --------------------------------------------------
# 3. Check the score volume
# --------------------------------------------------

assert sqrtL_score_zyx.shape == shape_zyx
assert np.isfinite(sqrtL_score_zyx).all()
assert np.all(sqrtL_score_zyx >= 0.0)
assert np.all(sqrtL_score_zyx <= 1.0)
assert np.allclose(sqrtL_score_zyx[target_zyx], 1.0)
assert np.all(sqrtL_score_zyx[~active_zyx] == 0.0)


# --------------------------------------------------
# 4. Save the score and its definition
# --------------------------------------------------

np.save(
    OUTPUT_DIR / "sqrtL_score_zyx.npy",
    sqrtL_score_zyx,
)

score_info = {
    "formula": "P_hit ** (1 / sqrt(L))",
    "array_order": "Z,Y,X",
    "valid_score_voxels": int(valid_score_voxels.sum()),
    "minimum_score": float(sqrtL_score_zyx.min()),
    "maximum_score": float(sqrtL_score_zyx.max()),
}

(OUTPUT_DIR / "score_info.json").write_text(
    json.dumps(score_info, indent=2),
    encoding="utf-8",
)

print(f"Valid score voxels: {valid_score_voxels.sum():,}")
print(
    "Score range: "
    f"[{sqrtL_score_zyx.min():.6f}, "
    f"{sqrtL_score_zyx.max():.6f}]"
)
print(f"Saved: {project_relative(OUTPUT_DIR / 'sqrtL_score_zyx.npy')}")

Valid score voxels: 1,604,329
Score range: [0.000000, 1.000000]
Saved: results/p28_330_HET_009/07_reconstruction/sqrtL_score_zyx.npy


In [7]:
# Cell 7 — Apply the score and raw-MIP intensity thresholds

# --------------------------------------------------
# 1. Normalize the raw-image MIP
# --------------------------------------------------

finite_raw = raw_mip_yx[np.isfinite(raw_mip_yx)]

if finite_raw.size == 0:
    raise ValueError("The raw MIP contains no finite values.")

intensity_low, intensity_high = np.quantile(
    finite_raw,
    RAW_MIP_QUANTILES,
)

normalized_intensity_yx = np.clip(
    (raw_mip_yx - intensity_low)
    / max(
        intensity_high - intensity_low,
        NUMERICAL_EPSILON,
    ),
    0.0,
    1.0,
).astype(np.float32)


# --------------------------------------------------
# 2. Apply both thresholds
# --------------------------------------------------

score_mask_zyx = sqrtL_score_zyx > SCORE_THRESHOLD
intensity_mask_yx = normalized_intensity_yx > INTENSITY_THRESHOLD

# Apply the same XY intensity gate at every Z slice.
candidate_mask_zyx = (
    active_zyx
    & score_mask_zyx
    & intensity_mask_yx[None, :, :]
)

# Retain the soma target independently of the thresholds.
candidate_mask_zyx |= target_zyx

assert candidate_mask_zyx.shape == shape_zyx
assert candidate_mask_zyx.dtype == bool
assert np.all(candidate_mask_zyx[target_zyx])
assert not np.any(candidate_mask_zyx & ~active_zyx)


# --------------------------------------------------
# 3. Save the normalized MIP and candidate mask
# --------------------------------------------------

np.save(
    OUTPUT_DIR / "normalized_raw_mip_yx.npy",
    normalized_intensity_yx,
)

np.save(
    OUTPUT_DIR / "candidate_mask_zyx.npy",
    candidate_mask_zyx,
)

intensity_filter_info = {
    "raw_mip_quantiles": [float(q) for q in RAW_MIP_QUANTILES],
    "normalization_lower_value": float(intensity_low),
    "normalization_upper_value": float(intensity_high),
    "score_threshold": float(SCORE_THRESHOLD),
    "intensity_threshold": float(INTENSITY_THRESHOLD),
    "intensity_gate": "same XY gate at every Z slice",
    "soma_targets_retained": True,
    "candidate_voxels": int(candidate_mask_zyx.sum()),
}

(OUTPUT_DIR / "intensity_filter_info.json").write_text(
    json.dumps(intensity_filter_info, indent=2),
    encoding="utf-8",
)

print(f"Score threshold: {SCORE_THRESHOLD}")
print(f"Normalized raw-MIP threshold: {INTENSITY_THRESHOLD}")
print(f"Candidate-mask voxels: {candidate_mask_zyx.sum():,}")
print(f"Saved results to: {project_relative(OUTPUT_DIR)}")

Score threshold: 0.5
Normalized raw-MIP threshold: 0.12
Candidate-mask voxels: 235,380
Saved results to: results/p28_330_HET_009/07_reconstruction


In [8]:
# Cell 8 — Remove small components and save the final mask

MIN_COMPONENT_VOXELS = 10


# --------------------------------------------------
# 1. Label the candidate components
# --------------------------------------------------

candidate_components_zyx, component_count_before = ndi.label(
    candidate_mask_zyx,
    structure=CONNECTIVITY_3D,
)

component_sizes = np.bincount(
    candidate_components_zyx.ravel()
)


# --------------------------------------------------
# 2. Select components to retain
# --------------------------------------------------

keep_component = component_sizes >= MIN_COMPONENT_VOXELS

# Exclude the background label.
keep_component[0] = False

# Always retain entire components containing soma-target voxels.
soma_component_ids = np.unique(
    candidate_components_zyx[target_zyx]
)

soma_component_ids = soma_component_ids[
    soma_component_ids > 0
]

keep_component[soma_component_ids] = True


# --------------------------------------------------
# 3. Construct the final mask
# --------------------------------------------------

final_mask_zyx = keep_component[candidate_components_zyx]
final_mask_zyx |= target_zyx

assert final_mask_zyx.shape == shape_zyx
assert final_mask_zyx.dtype == bool
assert np.all(final_mask_zyx[target_zyx])
assert not np.any(final_mask_zyx & ~candidate_mask_zyx)

removed_voxels = int(
    np.sum(candidate_mask_zyx & ~final_mask_zyx)
)


# --------------------------------------------------
# 4. Save the final mask and filtering record
# --------------------------------------------------

np.save(
    OUTPUT_DIR / "final_mask_zyx.npy",
    final_mask_zyx,
)

mask_info = {
    "component_filtering": True,
    "minimum_component_voxels": int(MIN_COMPONENT_VOXELS),
    "component_connectivity": 26,
    "soma_target_components_retained": True,
    "candidate_voxels": int(candidate_mask_zyx.sum()),
    "final_voxels": int(final_mask_zyx.sum()),
    "removed_voxels": removed_voxels,
    "components_before_filtering": int(component_count_before),
    "components_after_filtering": int(keep_component.sum()),
}

(OUTPUT_DIR / "mask_info.json").write_text(
    json.dumps(mask_info, indent=2),
    encoding="utf-8",
)

print(f"Minimum component size: {MIN_COMPONENT_VOXELS} voxels")
print(
    f"Components: {component_count_before:,} "
    f"→ {int(keep_component.sum()):,}"
)
print(f"Removed voxels: {removed_voxels:,}")
print(f"Final-mask voxels: {final_mask_zyx.sum():,}")
print(f"Saved: {project_relative(OUTPUT_DIR / 'final_mask_zyx.npy')}")

Minimum component size: 10 voxels
Components: 970 → 252
Removed voxels: 1,945
Final-mask voxels: 233,435
Saved: results/p28_330_HET_009/07_reconstruction/final_mask_zyx.npy


In [ ]:
# Cell 9 — Save reusable inputs and the reconstruction run record
#
# Save the existing arrays and settings for Step08 and Step09.
# Paths in the run record are relative to the repository root.

from importlib.metadata import PackageNotFoundError, version


# --------------------------------------------------
# 1. Require a converged reconstruction
# --------------------------------------------------

if not solver_info["converged"]:
    raise RuntimeError(
        "The hitting map has not converged. "
        "Complete Cells 1–8 before saving the run record."
    )


# --------------------------------------------------
# 2. Save the reusable inputs and ROI metadata
# --------------------------------------------------

np.save(OUTPUT_DIR / "signal_zyx.npy", signal_zyx)
np.save(OUTPUT_DIR / "active_domain_zyx.npy", active_zyx)
np.save(OUTPUT_DIR / "soma_target_zyx.npy", target_zyx)
np.save(OUTPUT_DIR / "raw_mip_yx.npy", raw_mip_yx)

metadata_table.to_csv(
    OUTPUT_DIR / "roi_metadata.csv",
    index=False,
)


# --------------------------------------------------
# 3. Check the complete set of saved reconstruction files
# --------------------------------------------------

saved_files = {
    "signal": "signal_zyx.npy",
    "active_domain": "active_domain_zyx.npy",
    "soma_target": "soma_target_zyx.npy",
    "raw_mip": "raw_mip_yx.npy",
    "normalized_raw_mip": "normalized_raw_mip_yx.npy",
    "hitting_probability": "P_hit_zyx.npy",
    "path_length": "L_zyx.npy",
    "adjusted_score": "sqrtL_score_zyx.npy",
    "candidate_mask": "candidate_mask_zyx.npy",
    "final_mask": "final_mask_zyx.npy",
    "roi_metadata": "roi_metadata.csv",
}

missing_files = [
    project_relative(OUTPUT_DIR / filename)
    for filename in saved_files.values()
    if not (OUTPUT_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing reconstruction files. Complete Cells 1–8 first:\n"
        + "\n".join(missing_files)
    )


# --------------------------------------------------
# 4. Record paths, reconstruction settings, and software versions
# --------------------------------------------------

manual_trace_file = (
    DATA_DIR
    / "manual_tracing"
    / f"{SAMPLE_NAME}_manual.traces"
)

package_versions = {}

for package_name in (
    "numpy",
    "pandas",
    "scipy",
    "scikit-image",
    "numba",
    "tqdm",
):
    try:
        package_versions[package_name] = version(package_name)
    except PackageNotFoundError:
        package_versions[package_name] = "not installed"

run_info = {
    "sample": SAMPLE_NAME,
    "path_base": "repository_root",
    "array_order_3d": "Z,Y,X",
    "array_order_2d": "Y,X",
    "shape_zyx": [int(n) for n in shape_zyx],
    "soma_center_x_local": float(soma_center_x_local),
    "soma_center_y_local": float(soma_center_y_local),
    "input_files": {
        name: project_relative(path)
        for name, path in INPUT_FILES.items()
    },
    "manual_trace_file": project_relative(manual_trace_file),
    "manual_trace_available": manual_trace_file.is_file(),
    "transition_weights": {
        "signal_power": float(SIGNAL_POWER),
        "epsilon": float(NUMERICAL_EPSILON),
    },
    "active_domain": active_domain_summary,
    "solver": solver_info,
    "path_length": path_length_info,
    "score": score_info,
    "intensity_filter": intensity_filter_info,
    "mask": mask_info,
    "files": saved_files,
    "environment": {
        "python": sys.version.split()[0],
        "packages": package_versions,
    },
}


# --------------------------------------------------
# 5. Save the run record
# --------------------------------------------------

run_info_file = OUTPUT_DIR / "run_info.json"

run_info_file.write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8",
)

print(f"Saved reconstruction files: {len(saved_files)}")
print(f"Run record: {project_relative(run_info_file)}")
print(f"Results directory: {project_relative(OUTPUT_DIR)}")
